# Assignment 2: Counter-Narrative Generation (worked solution, version 4, for the T4 GPU)

This notebook is a complete, heavily commented solution to **CT5146 Assignment 2**. It was written to be **studied**: every choice is explained and linked back to the course materials.

## What is new in version 4

Version 4 is version 3 with a **stricter rubric judge**. Everything else is unchanged: the data, the models, the prompts, generation, BLEU/ROUGE/BERTScore and retrieval.

- **Why:** in the version 3 run, the judge gave the top score (3 of 3) on every criterion to all 100 zero-shot replies, and 2.995 of 3 on average to the retrieval replies. A rubric that gives every reply full marks cannot show which replies, or which method, are better (a "ceiling effect").
- **What changed in the judge:**
  - **A finer scale:** 1–5 instead of 1–3, with a concrete meaning for every score.
  - **Sharper anchors.** For example, relevance 5 now requires a reply that refutes the specific claim with a concrete argument; a polite but generic "everyone deserves respect" reply gets 2.
  - **A calibration instruction:** "most replies are acceptable but not excellent: 5 is rare, 3 is acceptable".
  - **Critique first.** The judge writes one sentence naming the main weakness *before* giving the scores (chain-of-thought, L6 p.55). The critiques are saved with the outputs, so you can read *why* a reply got its scores.
  - **A score distribution table** shows how often each score was given, so you can check that the judge now uses the whole scale.
- **Runtime:** a little longer, because the judge now writes a sentence: about 70–100 minutes in total.

## What was new in version 3

Version 3 is version 2 **adjusted for the T4 GPU** (Colab's standard GPU, 15 GB usable memory). Nothing else changed.
- **Smaller models so that both fit on the T4:**
  - **Qwen2.5-3B-Instruct** writes the counter-narratives (version 2 used 7B).
  - **Qwen2.5-7B-Instruct** scores them with the rubric (version 2 used 14B).
  - Both are still loaded in 4-bit with exactly the `BitsAndBytesConfig` settings from the course lab (Lab7).
- **Number format:** the T4 cannot use `bfloat16`, so calculations use `float16`. The notebook detects this automatically.
- **Runtime:** about 60–90 minutes instead of 45–75.
- **Text:** model names, memory figures, the runtime and the example answers are updated to match.

Carried over unchanged from version 2:
- **Simpler, more standard code:**
  - `train_test_split` (scikit-learn) for the split;
  - `util.semantic_search` (Sentence Transformers) for retrieval;
  - `tqdm` progress bars;
  - the judge answers in JSON, read with `json.loads`.
  - There are no calculations inside f-strings, no regular expressions, no comprehensions and no deep nesting.
- **No manual scoring.** The LLM judge scores every reply, and the hand-scoring export from version 1 is gone.
- **Three versions of every written answer** (1c, 2c, 3c):
  - **A**: a short answer (max 50 words);
  - **B**: a long explanation without abbreviations;
  - **C**: a full worked example with dummy results and their interpretation.
- **A validation checklist** at the end covers code style, the brief, the course materials, and the tutor and classmate guidance.

## How to run it in Google Colab

1. **Runtime → Change runtime type → T4 GPU.**
2. **Runtime → Run all.**
3. **Expected runtime: about 70–100 minutes** with `N_TEST = 100`:
   - downloading the models (about 23 GB in total) takes roughly 5–10 minutes;
   - 2 × 100 replies are generated;
   - 2 × 100 replies are judged.

   For a quick first test, set `N_TEST = 20`.
4. Colab deletes files when the session ends. Download the generated CSV files from the **Files** panel if you want to keep them.
5. **When you are finished, choose Runtime → Disconnect and delete runtime**, so an idle session does not keep using your GPU time.

## How references to the course are written

| Short name | Course file |
|---|---|
| **W2** | `CT5146_Week2.pdf` (linguistic concepts) |
| **L3** | `CT5146-3.pdf` (vector space models, word embeddings) |
| **L4** | `4 Text Classification.pdf` |
| **L5** | `5 Language Models.pdf` |
| **L6** | `ct5146-6.pdf` (6a LLMs, 6b Prompting) |
| **L7** | `7 RAG and Generation.pdf` (7a RAG, 7b Generation) |
| **Lab7** | `07_transformer_Prompting_solution_2025_2026.ipynb` |
| **WS6b / WS7b** | Worksheets "6b Prompting" and "7b Generation" |
| **SK** | `starter_kit_counter_narrative.ipynb` (shared by the tutor) |
| **Tutor** | Assignment Q&A session with the tutor |
| **Peers** | Class chat (students relaying Q&A answers) |

For example, "L7 p.44" means page 44 of the RAG and Generation slides.

> **Note on the written answers (1c, 2c, 3c):**
> - The brief says they must be "written in your own words", so use the versions below to study and rewrite them yourself before submitting anything.
> - Version C uses the **real results of the version 3 run** for everything except the strict rubric, which is new in version 4. Its numbers are placeholders `[...]` until you run this notebook.

## Task roadmap (from the brief)

| # | Section | Focus | Marks |
|---|---|---|---:|
| 1 | Dataset & Task Understanding | Explore the hate-speech / counter-narrative dataset | 30 |
| 2 | Counter-Narrative Generation Method | Instruction-based LLM generation + evaluation | 40 |
| 3 | Improving Generation & Comparison | One improvement, compared with Section 2 | 30 |

## All design choices at a glance

Each choice is explained in detail in its section.

| Decision | Choice | Main reason (course source) |
|---|---|---|
| Hardware | **T4 GPU** (15 GB usable) | Colab's standard GPU; room for a 3B writer and a 7B judge in 4-bit; uses `float16` (no `bfloat16` on the T4) |
| Dataset | **Multitarget-CONAN** | Exactly the 3 required columns, 8 target groups, English, expert-written |
| Train/test split | `train_test_split` on the **unique hate speeches** (80/20, stratified by target) | No hate speech in both parts, so retrieval cannot "cheat"; the same target mix in both parts |
| Test subset | **100** hate speeches | The brief says "test subset"; full runs took classmates over 1.5 h |
| Generation LLM | **Qwen/Qwen2.5-3B-Instruct**, 4-bit | Open-source, instruction-tuned (L6 p.43), decoder-only (L3 p.91); clearly stronger than the lab models and fits the T4 |
| Quantisation | 4-bit NF4, the **Lab7 configuration** | Both models fit in GPU memory with only a small quality loss |
| Library | Hugging Face `transformers` pipeline | The same as Lab7; Ollama is allowed (Tutor) but awkward in Colab |
| Prompt | Starter-kit structure (Task / Rules / Comment) + persona and rules from AUTOMAT | SK; L6 pp.46–51, 58, 65–68 |
| Decoding | Greedy (`do_sample=False`), max 120 new tokens | Reproducible; less hallucination than sampling (L7 pp.44, 47–49; WS7b) |
| Reference metrics | BLEU, ROUGE-1/2/L, BERTScore | Named in the brief; BLEU code from Lab7; BERTScore measures meaning (L7 p.45) |
| Rubric | 4 criteria × **strict 1–5 scale**: relevance, tone, factuality, clarity | Own rubric allowed (tutor, peers); L7 p.41, p.43; "fact-bound" in the brief; a 1–3 scale hit the ceiling in the version 3 run |
| Rubric scorer | LLM judge **Qwen/Qwen2.5-7B-Instruct** (larger than the writer); **critique first**, then scores in **JSON** | "LLM can be asked to judge" (L7 p.27); chain-of-thought (L6 p.55); JSON output (L6 p.58); no hand scoring |
| Extra measure | **Refusal rate** | The tutor asked for it explicitly |
| Improvement (Section 3) | **Retrieval-augmented few-shot prompting**: the 3 most similar training pairs go into the prompt | Few-shot (L6 p.54) + Retrieve-Read RAG (L7 p.5) + dense retrieval (L7 pp.12–14) |
| Embedding model | `all-MiniLM-L6-v2` (Sentence Transformers) | Small and fast; sentence embeddings (L7 p.13); cosine similarity (L3 p.31) |

## Tiny Python refresher (for reading this notebook)

- **list**: `[1, 2, 3]`, an ordered collection. `my_list.append(x)` adds an item.
- **dict**: `{"name": "Ann", "age": 30}`, key → value pairs. Read a value with `my_dict["name"]`.
- **f-string**: `f"Hello {name}"` puts the value of the variable `name` into the text. Triple quotes `f"""..."""` allow several lines. In this notebook, f-strings only ever contain plain variable names.
  - `{ratio:.1f}` means "show with 1 decimal place".
- **function**: `def add(a, b): return a + b`, a reusable piece of code.
- **for loop**: `for item in my_list:` repeats the indented code once for every item.
  - `enumerate(my_list, start=1)` gives `(number, item)` pairs.
  - `zip(list_a, list_b)` walks through two lists side by side.
- **pandas DataFrame**: a table.
  - `df["COLUMN"]` is one column.
  - `df[condition]` keeps the rows where the condition is True.
  - `df.groupby("X")` makes groups of rows that share the same value in X.
- **`display(table)`** shows a table nicely in a notebook.
- **`tqdm(my_list)`** works like `my_list` in a for loop, but also shows a progress bar.

---
# Section 0: Setup

### Installing packages

Colab already includes `torch`, `transformers`, `accelerate`, `scikit-learn`, `pandas`, `matplotlib` and `tqdm`. We only add:

| Package | Why we need it |
|---|---|
| `bitsandbytes` | 4-bit loading of the LLMs (also installed in Lab7) |
| `evaluate` | Loads BLEU / ROUGE / BERTScore with one line each (the same library as Lab7) |
| `rouge_score`, `bert_score` | Needed by `evaluate` for ROUGE and BERTScore |
| `sentence-transformers` | Sentence embeddings for retrieval in Section 3 (L7 p.13) |

In [ ]:
# The "!" at the start means: run this as a terminal command, not as Python.
# "-q" means quiet (less output).
!pip install -q bitsandbytes evaluate rouge_score bert_score sentence-transformers

In [ ]:
# ----- Standard Python modules -----
import os        # to check whether a file exists
import json      # to read the judge's answers, which are written in JSON format
import random    # for reproducible random choices

# ----- Data analysis, plotting, progress bars, data splitting -----
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split

# ----- Deep learning / LLMs -----
import torch
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline, set_seed

# ----- Evaluation and retrieval -----
import evaluate
from sentence_transformers import SentenceTransformer, util

# Hide long warning messages from transformers (they are harmless, see the note in Section 2a).
transformers.logging.set_verbosity_error()

# Show long texts in tables without cutting them off too early.
pd.set_option("display.max_colwidth", 200)

### Settings

All important settings are collected in one cell, so you can change them in one place.

- **`RANDOM_SEED = 42`**: every split, sample and shuffle gives the same result each time (reproducible results).
- **`TEST_SIZE = 0.2`**: a standard 80/20 train/test split.
- **`N_TEST = 100`**: we generate for 100 test hate speeches, not all ~740.
  - The brief itself says "on a test subset", and classmates reported over 1.5 hours for full runs.
  - 100 is a compromise: big enough for stable averages, small enough to run both methods plus the judge in about 70–100 minutes on a T4.
- **`MAX_NEW_TOKENS = 120`**: the prompt asks for at most 60 words, which is roughly 80 tokens; 120 leaves a safety margin. The starter kit (SK) also used 120.
- **`JUDGE_MAX_NEW_TOKENS = 150`**: the judge writes a one-sentence critique plus the scores in JSON (about 60–100 tokens); 150 leaves a safety margin.
- **`K_EXAMPLES = 3`**: the number of retrieved examples in Section 3.
  - L6 p.54 says few-shot uses "a few (1–5) examples".
  - 3 gives variety without making the prompt too long ("overly long prompts can hurt", L6 p.49).

In [ ]:
RANDOM_SEED = 42

# Data
DATA_FILE = "Multitarget-CONAN.csv"   # used if you upload the file yourself
DATA_URL = (
    "https://raw.githubusercontent.com/marcoguerini/CONAN/master/"
    "Multitarget-CONAN/Multitarget-CONAN.csv"
)
TEST_SIZE = 0.2          # 20% of the unique hate speeches go to the test set
N_TEST = 100             # how many test hate speeches we generate for (use 20 for a quick try)

# Models (all open-source, no sign-up needed)
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"         # writes the counter-narratives
JUDGE_MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"   # scores them with the rubric (larger than the writer)
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"   # retrieval in Section 3

# Generation
MAX_NEW_TOKENS = 120         # maximum length of one generated counter-narrative
JUDGE_MAX_NEW_TOKENS = 150   # maximum length of one judge answer (critique + scores)
K_EXAMPLES = 3               # number of retrieved examples in Section 3

# Make everything reproducible
random.seed(RANDOM_SEED)
set_seed(RANDOM_SEED)    # sets the seed for Python, NumPy and PyTorch at once

### GPU check and number format

**The notebook needs a GPU.** 4-bit loading only works on a GPU.

**Number format: float16 on the T4.**
- Models do their calculations with 16-bit numbers to save memory and time.
- There are two 16-bit formats:
  - **bfloat16** ("brain float 16") has the same range as normal 32-bit numbers, so very large or very small values do not overflow. It needs a GPU of the "Ampere" generation or newer (compute capability 8 or higher), such as the L4 or A100.
  - **float16** is the older format. The **T4** (capability 7.5) supports only this one.
- The code checks the GPU and picks the right format automatically, so on the T4 it uses float16. If you ever get a newer GPU, the same code switches to bfloat16.

**What memory we need.** About 2 GB for the 3B writer + 5.5 GB for the 7B judge + 1.4 GB for the BERTScore model, so about 9 GB in total. The T4 has about 15 GB usable.

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError("No GPU found. Choose Runtime > Change runtime type > T4 GPU, then run again.")

gpu_name = torch.cuda.get_device_name(0)
gpu_memory_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3   # bytes -> GB
print("GPU:", gpu_name)
print("GPU memory (GB):", round(gpu_memory_gb, 1))

major_version, minor_version = torch.cuda.get_device_capability(0)
if major_version >= 8:              # L4, A100 and newer
    COMPUTE_DTYPE = torch.bfloat16
else:                               # older GPUs such as the T4
    COMPUTE_DTYPE = torch.float16
print("Number format for calculations:", COMPUTE_DTYPE)

if gpu_memory_gb < 14:
    print("WARNING: this GPU has less memory than a T4 (15 GB). The models may not fit.")
    print("Choose Runtime > Change runtime type > T4 GPU.")

---
# Section 1: Dataset & Counter-Narrative Task Understanding (30 marks)

## Choice of dataset: Multitarget-CONAN

The brief offers four example datasets (Multitarget-CONAN is listed twice) or "another appropriate publicly available counter-narrative dataset". I chose **Multitarget-CONAN** (Fanton et al., 2021) for these reasons:

1. **It already has exactly the required columns.** `HATE_SPEECH`, `COUNTER_NARRATIVE` and `TARGET` are the column names the brief asks for, so no mapping is needed.
2. **Several target groups (8):** MUSLIMS, MIGRANTS, WOMEN, LGBT+, JEWS, POC (people of colour), DISABLED and "other".
   - Section 1 asks us to analyse the target distribution, and with several targets that analysis is meaningful.
   - The original **CONAN** dataset is about **Islamophobia only**, so its target distribution would have just one class.
3. **English only.** Open LLMs and BERTScore (`lang="en"`) work best in English. The original CONAN is multilingual (English, French, Italian) and would need filtering.
4. **High-quality references.** The counter-narratives were written and checked with experts from NGOs. This matters because the human counter-narratives are our "gold standard" for evaluation.
5. **Size: about 5,000 pairs.** That is enough for a training pool for retrieval (Section 3) and a separate test set, and still small enough for fast analysis.

**Data source.**
- The code first looks for an uploaded file `Multitarget-CONAN.csv`. You can download it from the Google Drive link in the brief and upload it via the Files panel.
- If no file is uploaded, it downloads the same dataset from the authors' official GitHub repository.
- The extra columns `INDEX` (row number) and `VERSION` (the data-collection round, V1–V6) are not needed and are dropped.

**Loading method.** We use `pandas.read_csv`, as in Lab7. The brief's tutorial uses Hugging Face `load_dataset("csv", ...)` (as in WS6b). Both work; pandas is simpler for analysis and plotting.

In [ ]:
# Load the dataset: use an uploaded file if there is one, otherwise download it.
if os.path.exists(DATA_FILE):
    print("Loading the uploaded file:", DATA_FILE)
    df = pd.read_csv(DATA_FILE)
else:
    print("No uploaded file found - downloading from the official CONAN GitHub repository...")
    df = pd.read_csv(DATA_URL)

print("Shape (rows, columns):", df.shape)
print("Columns:", list(df.columns))

# Check that the three columns the assignment needs are present.
for column in ["HATE_SPEECH", "COUNTER_NARRATIVE", "TARGET"]:
    if column not in df.columns:
        print("WARNING: column", column, "is missing - rename your columns to match the brief.")

df.head()   # show the first 5 rows

### Cleaning

We keep the cleaning minimal on purpose. The texts are given to an LLM, which handles normal punctuation and capitalisation, so the heavy preprocessing from L3 p.70 (stopword removal, stemming, lowercasing) would only damage the text.

What we do:
- keep only the 3 needed columns;
- count and remove rows with missing values;
- remove spaces at the start and end of each text;
- remove exact duplicate rows (the same hate speech *and* the same counter-narrative twice).

In [ ]:
df = df[["HATE_SPEECH", "COUNTER_NARRATIVE", "TARGET"]].copy()   # keep only the needed columns

print("Missing values per column:")
print(df.isna().sum())
rows_before = len(df)

df = df.dropna()                                      # remove rows with empty cells
for column in ["HATE_SPEECH", "COUNTER_NARRATIVE", "TARGET"]:
    df[column] = df[column].astype(str).str.strip()   # remove leading/trailing spaces
df = df.drop_duplicates()                             # remove exact duplicate rows
df = df.reset_index(drop=True)                        # renumber the rows 0, 1, 2, ...

rows_after = len(df)
print("Rows before cleaning:", rows_before)
print("Rows after cleaning: ", rows_after)

### Train / test split, and why it is done "by hate speech"

The brief says: use the **training data** to build the method and the retrieval examples, and the **test data** only for final evaluation.

**Important detail: the same hate speech often appears several times, each time with a different counter-narrative.** In Multitarget-CONAN, 535 hate speeches have more than one counter-narrative; one has 55. If we split the *rows* randomly, the same hate speech could end up in both train and test. Then, in Section 3, the retrieval step would find the *identical* hate speech in the training data and copy its human answer. That is **data leakage**, and it would make the improvement look much better than it really is.

**Solution: split the list of unique hate speeches, not the rows.**
1. Make a table with one row per unique hate speech (and its target).
2. Split that table with scikit-learn's standard `train_test_split` (80% / 20%).
   - `stratify=...TARGET` keeps the same share of each target group in both parts.
   - `random_state` makes the split reproducible.
3. Every row of the full data goes to the side where its hate speech went.

In [ ]:
# One row per unique hate speech, together with its target group
unique_hate_speech = df.drop_duplicates(subset="HATE_SPEECH")[["HATE_SPEECH", "TARGET"]]

train_hate_speech, test_hate_speech = train_test_split(
    unique_hate_speech,
    test_size=TEST_SIZE,
    random_state=RANDOM_SEED,
    stratify=unique_hate_speech["TARGET"],    # same target mix in train and test
)

# .isin(...) is True for rows whose hate speech was put in the test part
is_test_row = df["HATE_SPEECH"].isin(test_hate_speech["HATE_SPEECH"])
test_df = df[is_test_row].reset_index(drop=True)
train_df = df[~is_test_row].reset_index(drop=True)     # "~" means NOT

n_train_rows = len(train_df)
n_test_rows = len(test_df)
n_train_unique = train_df["HATE_SPEECH"].nunique()
n_test_unique = test_df["HATE_SPEECH"].nunique()
print(f"Train: {n_train_rows} rows, {n_train_unique} unique hate speeches")
print(f"Test:  {n_test_rows} rows, {n_test_unique} unique hate speeches")

# Safety check: this must be 0, otherwise there is leakage.
overlap = set(train_df["HATE_SPEECH"]).intersection(set(test_df["HATE_SPEECH"]))
print("Hate speeches that appear in BOTH train and test:", len(overlap))

## 1a: Dataset (explore hate-speech targets and response characteristics) (10 marks)
*Brief: load your chosen dataset, count the occurrences of each target category, and compute summary statistics (lengths) of both hate speech and counter-narratives.*

What we compute, and why:
- **Target counts and percentages**: how balanced the target groups are.
- **Lengths in words and in characters.**
  - The tutor mentioned *character length* explicitly.
  - Words are easier to interpret.
  - Word counts split on spaces, which is a simple form of tokenization: they count "tokens" in the sense of W2 p.8.
- **Lengths per target group**: are some groups answered at more length?
- **Counter-narratives per hate speech**: shows the "one-to-many" nature of the task.
- **Tokens and types** (W2 p.8): tokens = all words; types = different words.
- **Share of counter-narratives containing numbers or web links**: a rough hint of how often experts use facts (the brief asks for "fact-bound" replies).

The analysis uses the **whole dataset** because it describes the data. The train/test sizes are printed above.

In [ ]:
# ---- Target distribution ----
target_counts = df["TARGET"].value_counts()                              # count per target
target_percent = df["TARGET"].value_counts(normalize=True) * 100         # share per target in %

target_table = pd.DataFrame({"count": target_counts, "percent": target_percent.round(1)})
n_targets = len(target_counts)
print("Number of different targets:", n_targets)
display(target_table)

In [ ]:
# ---- Text lengths ----
# .str.split() splits a text on spaces into a list of words; .str.len() then counts the words.
df["hate_speech_words"] = df["HATE_SPEECH"].str.split().str.len()
df["counter_narrative_words"] = df["COUNTER_NARRATIVE"].str.split().str.len()
# .str.len() directly on the text counts characters.
df["hate_speech_chars"] = df["HATE_SPEECH"].str.len()
df["counter_narrative_chars"] = df["COUNTER_NARRATIVE"].str.len()

length_columns = ["hate_speech_words", "counter_narrative_words",
                  "hate_speech_chars", "counter_narrative_chars"]
print("Summary statistics of lengths:")
display(df[length_columns].describe().round(1))

mean_hate_speech_words = df["hate_speech_words"].mean()
mean_counter_narrative_words = df["counter_narrative_words"].mean()
ratio = mean_counter_narrative_words / mean_hate_speech_words
print(f"On average, a counter-narrative is {ratio:.1f} times as long as the hate speech (in words).")

print("Average length in words per target group:")
per_target_lengths = df.groupby("TARGET")[["hate_speech_words", "counter_narrative_words"]].mean()
per_target_lengths = per_target_lengths.round(1)
display(per_target_lengths)

In [ ]:
# ---- One hate speech, several counter-narratives ----
replies_per_hate_speech = df.groupby("HATE_SPEECH").size()     # number of rows per hate speech
n_unique = len(replies_per_hate_speech)
n_with_several = (replies_per_hate_speech > 1).sum()
most_replies = replies_per_hate_speech.max()
print("Unique hate speeches:", n_unique)
print("Hate speeches with more than one counter-narrative:", n_with_several)
print("Most counter-narratives for one hate speech:", most_replies)


# ---- Tokens and types (W2 p.8) ----
def count_tokens_and_types(texts):
    """Return (number of tokens, number of types) for a list of texts.
    Tokens = all words. Types = different words."""
    tokens = []
    for text in texts:
        for word in text.lower().split():
            tokens.append(word)
    n_tokens = len(tokens)
    n_types = len(set(tokens))     # set() keeps each word only once
    return n_tokens, n_types


for column in ["HATE_SPEECH", "COUNTER_NARRATIVE"]:
    n_tokens, n_types = count_tokens_and_types(df[column])
    print(f"{column}: {n_tokens} tokens, {n_types} types")

# ---- How often do experts use numbers or links? (a rough sign of 'facts') ----
has_number = df["COUNTER_NARRATIVE"].str.contains("[0-9]")     # "[0-9]" = any digit
has_link = df["COUNTER_NARRATIVE"].str.contains("http")
share_with_number = has_number.mean() * 100                       # mean of True/False = share
share_with_link = has_link.mean() * 100
print(f"Counter-narratives containing a number: {share_with_number:.1f}%")
print(f"Counter-narratives containing a web link: {share_with_link:.1f}%")

## 1b: Text analysis and visualisation (10 marks)
*Brief: create two clear, labelled plots: a bar chart (or other diagram) of hate-target distributions, and a distribution comparison of input vs. response text lengths.*

- **Figure 1** is a bar chart of the target counts, with the number written above each bar.
- **Figure 2** has two panels:
  - (a) overlapping histograms of the word lengths of hate speech and counter-narratives;
  - (b) box plots of character lengths. A box plot shows the median (middle line), the middle 50% of the data (the box) and outliers (dots).
- **Figure 3** (extra) shows the average lengths per target group.

Every plot has a title, axis labels and, where needed, a legend ("clear, labelled").

In [ ]:
# ---- Figure 1: target distribution ----
plt.figure(figsize=(9, 4))
plt.bar(target_counts.index, target_counts.values, color="steelblue")
for position, value in enumerate(target_counts.values):
    plt.text(position, value + 10, str(value), ha="center")    # write the count above each bar
plt.title("Figure 1: Number of hate-speech / counter-narrative pairs per target group")
plt.xlabel("Target group")
plt.ylabel("Number of pairs")
plt.tight_layout()
plt.show()

In [ ]:
# ---- Figure 2: length of input (hate speech) vs response (counter-narrative) ----
fig, axes = plt.subplots(1, 2, figsize=(13, 4))    # 1 row, 2 panels side by side

# (a) histograms of word counts; alpha=0.6 makes the bars see-through so both are visible
axes[0].hist(df["hate_speech_words"], bins=40, range=(0, 120), alpha=0.6,
             label="Hate speech (input)")
axes[0].hist(df["counter_narrative_words"], bins=40, range=(0, 120), alpha=0.6,
             label="Counter-narrative (response)")
axes[0].set_title("(a) Length distribution in words")
axes[0].set_xlabel("Number of words")
axes[0].set_ylabel("Number of texts")
axes[0].legend()

# (b) box plots of character counts
axes[1].boxplot([df["hate_speech_chars"], df["counter_narrative_chars"]])
axes[1].set_xticks([1, 2], ["Hate speech (input)", "Counter-narrative (response)"])
axes[1].set_title("(b) Length in characters")
axes[1].set_ylabel("Number of characters")

fig.suptitle("Figure 2: Input vs. response text length")
plt.tight_layout()
plt.show()

In [ ]:
# ---- Figure 3 (extra): average length per target group ----
per_target_lengths.plot(kind="bar", figsize=(10, 4))
plt.title("Figure 3: Average length (words) of hate speech and counter-narrative per target")
plt.xlabel("Target group")
plt.ylabel("Average number of words")
plt.legend(["Hate speech", "Counter-narrative"])
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 1c: Explanations (10 marks)

**Question:** Explain your dataset selection and your data analysis process. Include: target distribution; text-length differences; one important observation from your visualisations; one challenge for counter-narrative generation. (Max **50 words**, in your own words.)

### Version A: short answer (47 words)

```
I chose Multitarget-CONAN (English, expert-written, eight targets). I counted targets and measured word/character lengths. Targets are imbalanced (Muslims 27%, disabled 4%); counter-narratives are twice as long as hate speech (25 versus 13 words). Many hate speeches have several valid replies, so generating specific, factual responses is challenging.
```

### Version B: long explanation (study notes)

I selected the Multitarget-CONAN dataset because it already contains the three fields the assignment requires: the hate speech, the human-written counter-narrative and the target group. No column mapping was needed. The dataset is in English, which suits open-source language models and the English model used by BERTScore. Its counter-narratives were written and reviewed together with experts from non-governmental organisations, so they are trustworthy reference answers for evaluation. Unlike the original CONAN dataset, which only covers Islamophobia, it covers eight target groups, so the target distribution can actually be analysed.

My analysis process had five steps:
1. I removed empty and duplicated rows. There were no empty values and one exact duplicate, leaving 5,002 pairs.
2. I split the data into a training part and a test part by unique hate speech, so that no hate speech appears in both parts.
3. I counted the pairs for each target group.
4. I measured the length of every text in words and in characters.
5. I plotted the target counts as a bar chart and compared the length distributions with histograms and box plots.

What I found:
- **The target groups are clearly imbalanced.** Muslims account for about 27 percent of the pairs, while disabled people account for only about 4 percent.
- **Counter-narratives are roughly twice as long** as the hate speech they answer: about 25 words compared with about 13 words on average, with a much wider spread of lengths.
- **Experts rarely use statistics.** Only about 4.5 percent of the expert replies contain a number, and almost none contain a web link; most argue with reasoning and shared values.
- **There is no single correct answer.** 535 hate speeches have more than one counter-narrative, and one of them has 55.

That last observation points to the main challenge for both generation and evaluation. A model has to produce a reply that is specific, factual and respectful. At the same time, comparing it with only one human reply can underestimate a good answer that is simply worded differently.

### Version C: full worked example with the real results and their interpretation

**Results from this run:**

| Target | Pairs | Share |
|---|---:|---:|
| MUSLIMS | 1,335 | 26.7% |
| MIGRANTS | 956 | 19.1% |
| WOMEN | 662 | 13.2% |
| LGBT+ | 617 | 12.3% |
| JEWS | 594 | 11.9% |
| POC | 352 | 7.0% |
| other | 266 | 5.3% |
| DISABLED | 220 | 4.4% |

| Length | Hate speech | Counter-narrative |
|---|---:|---:|
| Mean words | 13.2 | 24.8 |
| Median words | 12 | 23 |
| Standard deviation (words) | 7.8 | 11.4 |
| Longest text (words) | 68 | 149 |
| Mean characters | 75.5 | 140.7 |
| Median characters | 66 | 130 |

Other results:
- **Cleaning:** 5,003 rows loaded, no missing values, 1 exact duplicate removed, leaving 5,002 pairs.
- **Unique hate speeches:** 3,718. Of these, 535 have more than one counter-narrative; the maximum is 55.
- **Tokens / types:** hate speech 66,154 / 5,500; counter-narratives 123,916 / 10,803.
- **Expert replies with facts:** 4.5% contain a number and 0.2% contain a web link.
- **Split:** 4,052 training rows (2,974 unique hate speeches) and 950 test rows (744 unique hate speeches), with **0 overlap**. The test subset has 100 hate speeches.

**Interpretation:**
- **Imbalance (Figure 1).**
  - The largest group (Muslims, 1,335) is about six times the smallest (disabled people, 220).
  - In the 100-item test subset only a handful of items belong to the small groups, so per-target results for them are not reliable.
  - Small groups also have fewer training examples for retrieval in Section 3.
- **Length (Figure 2).**
  - Replies are 1.9 times as long as the attacks, and their lengths vary more (standard deviation 11.4 versus 7.8 words; the longest reply has 149 words).
  - Three quarters of the expert replies have 30 words or fewer, so the prompt's limit of "2 to 3 sentences, maximum 60 words" fits the human style.
- **One-to-many.** 535 hate speeches have several valid replies. We therefore evaluate against **all** references for each hate speech, and we expect word-overlap scores such as BLEU to be low even for good replies.
- **Facts.** Only 4.5% of expert replies contain a number and 0.2% a link. "Fact-bound" in this dataset therefore mostly means *not inventing facts* and arguing with reasoning and values, which is exactly what the prompt's fail-case rule asks for.
- **Vocabulary.** Replies use almost twice as many different words as the hate speech (10,803 versus 5,500 types), partly because they are longer. Their wording is varied, which again makes exact word overlap hard to achieve.

**Filled-in short answer:** Version A above already uses these real numbers.

---
# Section 2: Instruction-Based LLM Counter-Narrative Generation (40 marks)

## 2a: Build an instruction-based counter-narrative generator (20 marks)
*Brief: initialise an open-source instruction LLM and write a prompt template to generate non-aggressive, fact-bound counter-narratives on a test subset.*

### Choice of LLM

**What the brief requires:** open-source, **instruction-tuned**, inference only (no training).

**Why instruction-tuned and decoder-only (course arguments):**
- **Instruction tuning:** instruction-tuned models were fine-tuned on instruction–response pairs, so they "follow natural-language commands more reliably" (L6 p.43).
- **Decoder-only:**
  - Decoder-only models (GPT, LLaMA, …) generate text autoregressively, one token after another: P(X) = Π P(xᵢ | x₁…xᵢ₋₁) (L3 p.91; L5 p.12).
  - Encoder-only models like BERT are not suitable: "Generation is difficult", "Prompting is not possible" (L6 p.19).
- **Pretrained model:** training our own LLM is "not feasible" (L6 p.23), so we use a pretrained one (the "pre-train, prompt, predict" paradigm, L6 p.41).

**Candidates considered:**

| Model | Size | Where it appears | Verdict |
|---|---|---|---|
| `bigscience/bloom-560m` | 0.56B | WS6b, WS7b | ✗ A **base** model, not instruction-tuned |
| `bigscience/bloomz-560m` | 0.56B | Lab7 (default) | ✗ Too weak: in Lab7 it gave the same answer for every input, whatever the prompting method |
| `google/flan-t5-large` | 0.8B | Lab7 Part A | ✗ Encoder–decoder trained mostly on short-answer tasks; usually gives very short outputs |
| `TinyLlama/TinyLlama-1.1B-Chat-v1.0` | 1.1B | Lab7 option | Possible, but weak at following detailed rules |
| `Qwen/Qwen2.5-1.5B-Instruct` | 1.5B | Version 1 of this solution | Fits easily, but weaker at following detailed rules |
| **`Qwen/Qwen2.5-3B-Instruct`** | **3.1B** | (not in the course) | ✓ **Chosen for the T4** |
| `Qwen/Qwen2.5-7B-Instruct` | 7.6B | Version 2 of this solution (L4) | Stronger, but on a T4 there is then no room for a larger judge |

**Why Qwen2.5-3B-Instruct:**
- It is open-source, with no sign-up or access request. The licence is the Qwen Research licence (rather than Apache 2.0), which allows non-commercial use such as coursework.
- It is instruction-tuned with a chat format, and clearly better than the small course models at following detailed rules (persona, tone, length, "only accepted facts").
- In 4-bit it needs only about 2 GB of GPU memory. That leaves room on the T4 for a **larger** judge model (7B), which the evaluation needs (see 2b).
- The tutor said "a small (low-parameter) model is fine", and a 3B model is small and fast on a T4.

**4-bit quantisation (the Lab7 configuration).**
- Normally every model weight is stored as a 16-bit number. **4-bit quantisation** stores each weight in 4 bits instead, about **4 times less memory**, with only a small loss of quality.
- **NF4** ("NormalFloat 4") is a 4-bit format designed for how model weights are distributed.
- **Double quantisation** also compresses the scaling constants, which saves a little more memory.
- Calculations are still done in 16 bits (`float16` on the T4).
- These are exactly the settings used in Lab7 (`BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_use_double_quant=True, bnb_4bit_quant_type="nf4", ...)`). WS6b used the 8-bit version of the same idea.

**Why the Hugging Face pipeline and not Ollama:** the tutor allowed Ollama, but it needs a separate server process, which is awkward in Colab. The `transformers` pipeline is exactly what Lab7 uses.

In [ ]:
# 4-bit loading settings, the same as in Lab7
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                      # store weights in 4 bits (about 4x less memory)
    bnb_4bit_use_double_quant=True,         # also compress the scaling constants
    bnb_4bit_quant_type="nf4",              # "NormalFloat 4", a 4-bit format made for model weights
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,   # calculations in 16 bits (float16 on the T4)
)


def load_llm(model_name):
    """Load a tokenizer + 4-bit model and wrap them in a text-generation pipeline (as in Lab7)."""
    print("Loading", model_name, "in 4-bit ...")
    llm_tokenizer = AutoTokenizer.from_pretrained(model_name)
    llm_model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map="auto",                  # put the model on the GPU automatically
    )
    llm_pipeline = pipeline("text-generation", model=llm_model, tokenizer=llm_tokenizer)
    return llm_tokenizer, llm_pipeline


tokenizer, generator = load_llm(MODEL_NAME)

memory_used_gb = torch.cuda.memory_allocated() / 1024**3
print(f"Model loaded. GPU memory in use: {memory_used_gb:.1f} GB")

### The chat template: the "hidden tags" from L6 p.43

L6 p.43 says that instruction-tuned models use **hidden tags** to mark where each message starts and ends, and that "Chat LLMs do not do next word completion on input!". The tokenizer's `apply_chat_template` adds these tags for us.

The cell below prints what the model *really* sees for a tiny example:
- The special markers (`<|im_start|>` and `<|im_end|>` for Qwen) separate the **system** message (instructions about behaviour), the **user** message (our prompt) and the place where the **assistant** answer starts.
- `add_generation_prompt=True` adds that final "assistant starts here" marker.

This is better than the plain-string prompts in Lab7, which did not use the template.

In [ ]:
example_messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Say hello."},
]
example_text = tokenizer.apply_chat_template(example_messages, tokenize=False,
                                             add_generation_prompt=True)
print(example_text)

### Prompt design

The prompt follows the **starter kit (SK)** structure (Task → Rules → Comment → "Counter-narrative:"), improved with the prompting guidelines from L6. Each part has a reason:

| Prompt part | What it says | Course source |
|---|---|---|
| **System message (persona)** | "You are an experienced counter-speech operator at an anti-discrimination NGO…" | AUTOMAT **A**ct as… (L6 p.66); a role influences style (L6 p.58) |
| **Task with a clear verb** | "Write a counter-narrative response to…" | Instruction with a verb (L6 p.48); AUTOMAT **T**argeted action |
| **Target group** | "The comment targets Muslims." | Context element (L6 p.46): helps the model address the right group |
| **Rules** | Respond to the specific claim; calm and non-aggressive; only widely accepted facts | The brief's "non-aggressive, fact-bound"; AUTOMAT **M**ode/tone; SK rules |
| **Fail case** | "If you are not sure a fact is correct, do not give numbers; argue from shared values and empathy instead" | Give fail cases (L6 p.51); AUTOMAT **A**typical cases; reduces hallucination (L7 p.43) |
| **Positive wording** | "Use neutral, respectful words" instead of only "don't…" | LLMs tend to do what they are told *not* to do (L6 p.51) |
| **Length limit** | "2 to 3 sentences (maximum 60 words)" | Be precise ("2–3 sentences", not "briefly", L6 p.50); matches the human length (~25 words) |
| **Output indicator** | "Provide only the counter-narrative…" + ending with "Counter-narrative:" | Output format / output indicator (L6 pp.46, 58); WS6b step 2 "specify the output" |

**Zero-shot:** no examples are given. This is our Section 2 baseline (L6 p.53; L3 p.93). The SK also calls itself "a zero-shot baseline".

**Shared rules:** the rules are stored in one variable `RULES` that Section 3 reuses unchanged, so the *only* difference between the two methods is the added examples.

In [ ]:
# Nicer wording for the target labels used in the dataset
TARGET_DESCRIPTIONS = {
    "MUSLIMS": "Muslims",
    "MIGRANTS": "migrants",
    "WOMEN": "women",
    "LGBT+": "LGBT+ people",
    "JEWS": "Jewish people",
    "POC": "people of colour",
    "DISABLED": "disabled people",
    "other": "a group of people",
}

# System message = persona and general behaviour (AUTOMAT "Act as" and "Mode")
SYSTEM_PROMPT = (
    "You are an experienced counter-speech operator at an anti-discrimination NGO. "
    "You write calm, respectful replies to hateful online comments. "
    "Your replies correct false claims with widely accepted facts, "
    "appeal to empathy and shared values, and never insult anyone."
)

# Rules shared by Section 2 and Section 3 (so the only difference in Section 3 is the examples)
RULES = """Rules:
- Respond directly to the specific claim made in the comment.
- Be calm, respectful and non-aggressive. Use neutral, respectful words (no slurs or insults).
- Only use facts that are widely accepted.
  If you are not sure a fact is correct, do not give numbers; argue from shared values and empathy instead.
- Write 2 to 3 sentences (maximum 60 words).
- Provide only the counter-narrative, without any introduction or label."""


def build_zero_shot_prompt(hate_speech, target):
    """Build the zero-shot user prompt for one hate speech (Section 2)."""
    target_text = TARGET_DESCRIPTIONS.get(target, "a group of people")   # .get: default if unknown
    prompt = f"""Task: Write a counter-narrative response to the hateful comment below.
The comment targets {target_text}.

{RULES}

Comment:
{hate_speech}

Counter-narrative:"""
    return prompt


print(build_zero_shot_prompt("Example hateful comment.", "MIGRANTS"))

### Generation settings and helper functions

**Decoding: greedy (`do_sample=False`).**
- Greedy decoding always takes the most probable next token. It is the `torch.argmax` route in WS7b; sampling is the `torch.multinomial` route.
- **Reproducible:** the same input always gives the same output, so Section 2 and Section 3 are compared fairly.
- **Less hallucination:** "increasing diversity (temperature) leads to more hallucinations" (L7 p.44). The brief wants fact-bound replies.
- **Downside:** greedy output can be repetitive or bland (L7 p.47).
- The SK used `temperature=0.7` (sampling). We deliberately do not, for the reasons above.
- **Harmless warning:** the model's default settings include a temperature. With `do_sample=False` it is ignored, and transformers may warn about this (Lab7 shows the same warning). We hid these warnings in the imports cell.

**Helper functions:**
- **`generate_text(...)`** builds the chat messages, applies the chat template and runs the pipeline.
  - `return_full_text=False` returns only the *new* text, so we don't need Lab7's `.replace(prompt, "")` trick.
  - The same function is reused for the judge later.
- **`clean_output(...)`** removes leftovers such as a repeated "Counter-narrative:" label or quotation marks. This is the same idea as Lab7's output cleaning.
- **`is_refusal(...)`** detects refusals. The tutor said: if the LLM refuses because of its safety training, **record the refusal rate**.
  - It is a simple keyword check on the start of the reply ("I can't help", "As an AI", …).
  - We avoid plain "I'm sorry", because real counter-narratives often start with "I'm sorry you feel that way".
  - An empty output also counts as a failure.

In [ ]:
def generate_text(llm_tokenizer, llm_pipeline, system_prompt, user_prompt, max_new_tokens):
    """Send one prompt to an instruction-tuned LLM and return only the generated answer."""
    messages = []
    if system_prompt is not None:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": user_prompt})

    # Add the model's hidden chat tags (L6 p.43)
    prompt_text = llm_tokenizer.apply_chat_template(messages, tokenize=False,
                                                    add_generation_prompt=True)

    outputs = llm_pipeline(
        prompt_text,
        max_new_tokens=max_new_tokens,
        do_sample=False,          # greedy decoding (see explanation above)
        return_full_text=False,   # only return the new text, not the prompt
    )
    answer = outputs[0]["generated_text"]
    return answer.strip()


def clean_output(text):
    """Remove labels/quotes the model sometimes adds around its answer."""
    text = text.strip()
    for label in ["Counter-narrative:", "Counter-Narrative:", "Counter narrative:", "Response:"]:
        if text.startswith(label):
            text = text[len(label):]          # cut the label off the front
            text = text.strip()
    text = text.strip('"“”')                  # remove straight or curly quotes at the start/end
    return text.strip()


REFUSAL_PHRASES = [
    "i can't help", "i cannot help", "i can't assist", "i cannot assist",
    "i can't provide", "i cannot provide", "i can't create", "i cannot create",
    "i can't fulfill", "i cannot fulfill", "i can't respond", "i cannot respond",
    "i can't engage", "i cannot engage", "i won't", "i will not",
    "as an ai", "i'm unable to", "i am unable to", "i'm not able to", "i am not able to",
]


def is_refusal(text):
    """Return True if the reply looks like a refusal (or is empty)."""
    if text.strip() == "":
        return True
    start = text[:150].lower()             # only look at the beginning of the reply
    start = start.replace("’", "'")        # treat curly apostrophes like straight ones
    for phrase in REFUSAL_PHRASES:
        if phrase in start:
            return True
    return False

### The test subset

From the test hate speeches we draw `N_TEST` at random with the standard pandas `.sample(...)`. For each one we keep:
- the hate speech;
- its target;
- **all** of its human counter-narratives as references. Several references are fairer for BLEU, ROUGE and BERTScore, because there are many valid ways to reply (see 1c).

The same subset is used in Section 2 and Section 3, so the comparison is fair.

In [ ]:
n_items = min(N_TEST, len(test_hate_speech))
test_subset = test_hate_speech.sample(n=n_items, random_state=RANDOM_SEED)

test_items = []
for hate_speech, target in zip(test_subset["HATE_SPEECH"], test_subset["TARGET"]):
    rows_for_this_hate_speech = test_df[test_df["HATE_SPEECH"] == hate_speech]
    item = {
        "hate_speech": hate_speech,
        "target": target,
        "references": list(rows_for_this_hate_speech["COUNTER_NARRATIVE"]),   # all human replies
    }
    test_items.append(item)

print("Number of test items used:", len(test_items))
print("Example item:", test_items[0])

### Sanity check on one example

Always try one example before running everything (WS6b does the same). If the output is empty or looks like garbage, something is wrong; check the GPU check cell first.

In [ ]:
first_item = test_items[0]
first_prompt = build_zero_shot_prompt(first_item["hate_speech"], first_item["target"])
first_answer = generate_text(tokenizer, generator, SYSTEM_PROMPT, first_prompt, MAX_NEW_TOKENS)
first_output = clean_output(first_answer)

print("HATE SPEECH:     ", first_item["hate_speech"])
print("TARGET:          ", first_item["target"])
print("GENERATED:       ", first_output)
print("HUMAN REFERENCE: ", first_item["references"][0])
print("Refusal?         ", is_refusal(first_output))

### Generate counter-narratives for the whole test subset (zero-shot)

A simple `for` loop generates for each item one by one (as in SK and Lab7). `tqdm` shows a progress bar with the elapsed time. The outputs are saved to a CSV file (an SK idea): if you need to evaluate again later, you can reload them instead of generating again.

In [ ]:
def make_results_table(items, predictions):
    """Put inputs, references and predictions in one DataFrame (one row per test item)."""
    rows = []
    for item, prediction in zip(items, predictions):
        rows.append({
            "hate_speech": item["hate_speech"],
            "target": item["target"],
            "human_reference": item["references"][0],
            "number_of_references": len(item["references"]),
            "prediction": prediction,
            "refusal": is_refusal(prediction),
        })
    return pd.DataFrame(rows)


baseline_predictions = []
for item in tqdm(test_items, desc="Zero-shot generation"):
    prompt = build_zero_shot_prompt(item["hate_speech"], item["target"])
    answer = generate_text(tokenizer, generator, SYSTEM_PROMPT, prompt, MAX_NEW_TOKENS)
    baseline_predictions.append(clean_output(answer))

baseline_df = make_results_table(test_items, baseline_predictions)
baseline_df.to_csv("section2_zero_shot_outputs.csv", index=False)
print("Saved section2_zero_shot_outputs.csv")
display(baseline_df[["hate_speech", "prediction", "human_reference"]].head(5))

## 2b: Evaluate counter-narratives (10 marks)
*Brief: evaluate your generated responses against human ground truths using quantitative NLP metrics like BLEU, ROUGE and BERTScore. Also evaluate using rubric-based scores on the generated counter-narratives.*

The tutor described **two kinds of evaluation**: **reference-based** (compare with the human counter-narrative) and **rubric-based** (a scale, applied through a prompt). We do both, plus the refusal rate.

### Part 1: Reference-based metrics

| Metric | What it measures | Course link |
|---|---|---|
| **BLEU** | Word n-gram (1–4 word sequences) **precision** against the references, with a penalty for too-short outputs. Exact word overlap. | Lab7 uses `evaluate.load("bleu")`; precision (L4 p.31) |
| **ROUGE-1 / ROUGE-2** | Overlap of single words / word pairs; **recall-oriented** (`evaluate` reports the F-measure) | Recall / F1 (L4 pp.31–32) |
| **ROUGE-L** | The **longest common subsequence**: shared words in the same order | n/a |
| **BERTScore-F1** | Embeds both texts with a BERT-style model, matches each word to its most similar word in the other text by **cosine similarity**, and reports P/R/F1. Measures **meaning**, so paraphrases score well. | Cosine (L3 p.31); BERT contextual embeddings (L3 p.85); the tutor's "semantic scores" |

**Why all four:**
- BLEU and ROUGE only count identical words. L7 p.45 warns that such metrics "focus on text fluency, not factual accuracy".
- A good counter-narrative can use completely different words from the human one, so BERTScore adds a *meaning-based* view.
- Even so, all of them only compare with the reference. That is why we also need the rubric.

**Details:**
- **Lowercasing:** BLEU is case-sensitive, so we lowercase both sides, as Lab7 did. ROUGE lowercases by itself.
- **Multiple references:** each prediction is compared with *all* human counter-narratives for that hate speech.
- **`rescale_with_baseline=True`** for BERTScore: raw BERTScores are all squeezed between about 0.8 and 0.9. Rescaling spreads them out so differences are easier to see. 0 then means "as similar as two unrelated sentences", and values can be slightly negative.
- **Expected values:** BLEU will be **low** (often below 0.05). That is normal for open-ended generation, because there are many valid replies (1c).

In [ ]:
bleu_metric = evaluate.load("bleu")
rouge_metric = evaluate.load("rouge")
bertscore_metric = evaluate.load("bertscore")


def lowercase_all(texts):
    """Return a new list with every text in lowercase."""
    lowered = []
    for text in texts:
        lowered.append(text.lower())
    return lowered


def compute_reference_scores(predictions, items):
    """Compute BLEU, ROUGE and BERTScore against all human references.
    Returns (dictionary of average scores, list of per-example BERTScore F1)."""
    references = []           # one list of human replies per test item
    lower_references = []     # the same, in lowercase (for BLEU)
    for item in items:
        references.append(item["references"])
        lower_references.append(lowercase_all(item["references"]))

    bleu_result = bleu_metric.compute(predictions=lowercase_all(predictions),
                                      references=lower_references)
    rouge_result = rouge_metric.compute(predictions=predictions, references=references)
    bert_result = bertscore_metric.compute(predictions=predictions, references=references,
                                           lang="en", rescale_with_baseline=True)

    per_example_bertscore = bert_result["f1"]      # one F1 value per test item
    scores = {
        "BLEU": bleu_result["bleu"],
        "ROUGE-1": rouge_result["rouge1"],
        "ROUGE-2": rouge_result["rouge2"],
        "ROUGE-L": rouge_result["rougeL"],
        "BERTScore-F1": sum(per_example_bertscore) / len(per_example_bertscore),
    }
    return scores, per_example_bertscore


baseline_reference_scores, baseline_bertscores = compute_reference_scores(baseline_predictions,
                                                                          test_items)
baseline_refusal_rate = baseline_df["refusal"].mean()     # mean of True/False = share of True

print("Reference-based scores (Section 2):")
for name, value in baseline_reference_scores.items():
    print(" ", name, round(value, 4))
print("  Refusal rate", round(baseline_refusal_rate, 3))

### Part 2: Rubric-based scores (LLM as a judge, strict 1–5 version)

**The rubric.** The tutor and classmates reported that you may **define your own rubric**, as long as you explain what each score means; classmates reported that a 1–3 scale was accepted. In the version 3 run, however, a 1–3 scale gave **every** zero-shot reply full marks, so it could not separate good from weak replies. Version 4 therefore uses a **1–5 scale with sharp anchors**:

| Score | relevance | tone | factuality | clarity |
|---|---|---|---|---|
| **5** | Directly refutes the specific claim with a concrete argument or example | Calm, respectful and empathetic; not preachy | Every statement accurate, or clearly presented as opinion or values | Clear, concise and well structured |
| **4** | Addresses the specific claim, but with a general argument | Respectful but somewhat lecturing or preachy | Accurate overall, but includes a broad generalisation | Clear, with minor awkwardness |
| **3** | Partly addresses the claim; mostly a general defence of the group | Noticeably moralising or condescending | Vague claims presented as facts that cannot be checked | Understandable but wordy or awkward |
| **2** | Generic statement about respect or tolerance that ignores the claim | Harsh or sarcastic | At least one doubtful claim or statistic | Hard to follow |
| **1** | Off-topic, or agrees with or repeats the hateful claim | Insulting or aggressive | Clearly false or invented facts or statistics | Incoherent or ungrammatical |
| **Source** | Peers/tutor: "does it actually counter the hate speech" | Brief: "non-aggressive"; style (L7 p.41) | Brief: "fact-bound"; hallucination (L7 p.43) | Fluency (L7 p.41) |

**Who scores?** The tutor allowed a prompt-based scale (an LLM judge), human scoring, or both. "LLM can be asked to judge many things!" (L7 p.27). We use **only the LLM judge**, so that all 200 replies (100 × 2 methods) are scored the same way and nothing has to be scored by hand.

**Choices for the judge:**
- **Qwen2.5-7B-Instruct, more than twice the size of the generator (3B).**
  - Judging is harder than writing a short reply, so the judge should be the stronger model.
  - A model that judges its own outputs tends to favour them (self-preference bias). Using a separate, larger model reduces this. It is still the same model family, so some bias may remain.
  - **This bias affects Section 2 and Section 3 equally,** because both use the same writer. The *comparison* stays fair.
- **Strictness against the "ceiling effect".** Language-model judges tend to be lenient: they like fluent, polite text and give it top marks. Three measures counter this:
  1. **Sharper anchors.** Every score has a concrete meaning. A polite but generic "everyone deserves respect" reply now gets relevance 2, not the top score.
  2. **A calibration instruction.** The prompt says that most replies are acceptable but not excellent: 5 is rare, 3 is acceptable.
  3. **Critique first.** The judge first writes one sentence naming the main weakness of the reply, and only then gives the scores. Reasoning before answering is chain-of-thought prompting (L6 p.55), and asking for a weakness makes the judge look critically. The critiques are saved, so you can check *why* a reply got its scores.
- **One reply at a time ("pointwise")**: the judge never sees the two methods side by side, so the order of the answers cannot bias it.
- **Reference-free**: the judge does not see the human reply. The rubric measures quality *by itself*; similarity to the reference is already covered by Part 1.
- **JSON output**, as recommended "for downstream processing" (L6 p.58): `{"critique": ..., "relevance": ..., "tone": ..., "factuality": ..., "clarity": ...}`.
  - The critique comes first, so the model writes its reasoning before the numbers.
  - The answer is read with the standard `json.loads`. Answers that cannot be read are counted and left out of the averages.
- **Greedy decoding** makes the scores reproducible.
- **A score distribution table** (after the Section 2 results) shows how often each score from 1 to 5 was given. If the judge still gave almost only 5s, the rubric would again be at the ceiling.

In [ ]:
judge_tokenizer, judge_generator = load_llm(JUDGE_MODEL_NAME)
memory_used_gb = torch.cuda.memory_allocated() / 1024**3
print(f"Judge loaded. GPU memory in use (both models): {memory_used_gb:.1f} GB")

# The strict rubric: criterion -> what 5, 4, 3, 2 and 1 mean
RUBRIC = {
    "relevance": "5 = directly refutes the specific claim with a concrete argument or example, "
                 "4 = addresses the specific claim but with a general argument, "
                 "3 = partly addresses the claim, mostly a general defence of the group, "
                 "2 = generic statement about respect or tolerance that ignores the claim, "
                 "1 = off-topic, or agrees with or repeats the hateful claim",
    "tone": "5 = calm, respectful and empathetic, not preachy, "
            "4 = respectful but somewhat lecturing or preachy, "
            "3 = noticeably moralising or condescending, "
            "2 = harsh or sarcastic, "
            "1 = insulting or aggressive",
    "factuality": "5 = every statement is accurate, or clearly presented as opinion or values, "
                  "4 = accurate overall but includes a broad generalisation, "
                  "3 = vague claims presented as facts that cannot be checked, "
                  "2 = at least one doubtful claim or statistic, "
                  "1 = clearly false or invented facts or statistics",
    "clarity": "5 = clear, concise and well structured, "
               "4 = clear with minor awkwardness, "
               "3 = understandable but wordy or awkward, "
               "2 = hard to follow, "
               "1 = incoherent or ungrammatical",
}

VALID_SCORES = [1, 2, 3, 4, 5, "1", "2", "3", "4", "5"]   # numbers, or numbers written as text

# The exact answer format we ask for (a normal string, so the curly brackets need no escaping).
# "critique" comes first, so the judge writes its reasoning before the scores.
JUDGE_JSON_FORMAT = ('{"critique": "<one sentence>", "relevance": <1-5>, "tone": <1-5>, '
                     '"factuality": <1-5>, "clarity": <1-5>}')


def build_judge_prompt(hate_speech, counter_narrative):
    """Build the prompt that asks the judge to critique and score one counter-narrative."""
    rubric_lines = ""
    for criterion, description in RUBRIC.items():
        rubric_lines += f"- {criterion}: {description}\n"

    prompt = f"""You are a strict expert evaluator of counter-narratives (replies to hate speech).
Most replies are acceptable but not excellent, so use the whole scale:
5 = excellent (rare), 4 = good, 3 = acceptable, 2 = weak, 1 = very poor.

Hateful comment:
{hate_speech}

Counter-narrative to evaluate:
{counter_narrative}

Scoring guide:
{rubric_lines}
First write one sentence of critique that names the main weakness of the counter-narrative.
Then give the scores. Answer with JSON only, in exactly this format:
{JUDGE_JSON_FORMAT}"""
    return prompt


def parse_judge_answer(judge_answer):
    """Read the judge's JSON answer. Returns (scores, critique).
    Missing or invalid scores become None."""
    scores = {}
    for criterion in RUBRIC:
        scores[criterion] = None                  # default: no valid score found
    critique = ""

    start = judge_answer.find("{")                # position of the first "{"
    end = judge_answer.rfind("}")                 # position of the last "}"
    if start == -1 or end == -1:
        return scores, critique                   # no JSON in the answer

    json_text = judge_answer[start:end + 1]       # cut out the part between { and }
    try:
        data = json.loads(json_text)
    except json.JSONDecodeError:
        return scores, critique                   # the JSON was broken
    if not isinstance(data, dict):
        return scores, critique

    critique = str(data.get("critique", ""))
    for criterion in RUBRIC:
        value = data.get(criterion)
        if value in VALID_SCORES:
            scores[criterion] = int(value)
    return scores, critique


def judge_all(items, predictions):
    """Ask the judge to critique and score every prediction.
    Returns (DataFrame of scores with rows = items and columns = criteria, list of critiques)."""
    all_scores = []
    all_critiques = []
    n_failed = 0
    for i in tqdm(range(len(items)), desc="Judging"):
        prompt = build_judge_prompt(items[i]["hate_speech"], predictions[i])
        answer = generate_text(judge_tokenizer, judge_generator, None, prompt, JUDGE_MAX_NEW_TOKENS)
        scores, critique = parse_judge_answer(answer)
        if None in scores.values():
            n_failed += 1
        all_scores.append(scores)
        all_critiques.append(critique)
    print("Judge answers that could not be read completely:", n_failed)
    judgements = pd.DataFrame(all_scores, dtype=float)    # None becomes NaN ("missing")
    return judgements, all_critiques


def average_rubric_scores(judgements):
    """Average of each criterion (missing values are skipped), plus the overall average."""
    criterion_averages = judgements.mean()             # one average per column
    averages = {}
    for criterion in RUBRIC:
        averages["Rubric " + criterion] = criterion_averages[criterion]
    averages["Rubric overall (1-5)"] = criterion_averages.mean()
    return averages


def score_distribution(judgements):
    """Table: how often each score (1-5) was given for each criterion."""
    counts_per_criterion = {}
    for criterion in RUBRIC:
        counts_per_criterion[criterion] = judgements[criterion].value_counts()
    distribution = pd.DataFrame(counts_per_criterion)       # rows = scores, columns = criteria
    distribution = distribution.sort_index().fillna(0).astype(int)
    distribution.index = distribution.index.astype(int)     # show 3 instead of 3.0
    distribution.index.name = "score"
    return distribution


# Show the judge prompt once, so you can see exactly what the judge receives
print(build_judge_prompt(test_items[0]["hate_speech"], baseline_predictions[0]))

In [ ]:
baseline_judgements, baseline_critiques = judge_all(test_items, baseline_predictions)

# Add the per-example rubric scores and critiques to the results table and save it again
for criterion in RUBRIC:
    baseline_df["rubric_" + criterion] = baseline_judgements[criterion]
baseline_df["rubric_critique"] = baseline_critiques
baseline_df.to_csv("section2_zero_shot_outputs.csv", index=False)
rubric_columns = ["prediction", "rubric_relevance", "rubric_tone", "rubric_factuality",
                  "rubric_clarity", "rubric_critique"]
display(baseline_df[rubric_columns].head(5))

In [ ]:
# ---- All Section 2 results in one table ----
section2_scores = {}
section2_scores.update(baseline_reference_scores)              # BLEU, ROUGE, BERTScore
section2_scores["Refusal rate"] = baseline_refusal_rate
section2_scores.update(average_rubric_scores(baseline_judgements))

section2_table = pd.DataFrame({"Section 2: zero-shot": section2_scores})
display(section2_table.round(3))

# ---- Does the judge use the whole scale? (checks for the "ceiling effect") ----
print("How often each score was given (Section 2, rows = score 1-5):")
display(score_distribution(baseline_judgements))

### How to read the Section 2 results

- **BLEU and ROUGE-2 will be low.**
  - The model rarely uses the same word sequences as the human expert, even when its reply is good.
  - This is the limitation from L7 p.45, made worse by the one-to-many nature of the data (1c).
- **BERTScore** is the better reference-based measure of whether the *meaning* is similar.
- **The rubric** says whether the reply is good *by itself*:
  - **tone** is usually high for instruction-tuned models;
  - **relevance** and **factuality** are where weaknesses show up, for example generic "everyone deserves respect" replies;
  - on the strict 1–5 scale, expect most scores around 3–4. If almost every score is 5 (see the distribution table), the judge is again at the ceiling.
- **The refusal rate** shows whether safety training stopped the model from answering.
- Read a few replies yourself (the table above). Numbers alone can hide problems.

## 2c: Explanation (10 marks)

**Question:** Briefly explain your instruction-based LLM generation method and the results. Include: why you chose the LLM; how you designed the prompt; which evaluation scores you used and why; one strength and one limitation of the generated counter-narratives. (Max **50 words**, in your own words.)

### Version A: short answer (44 words)

Fill in the two `[...]` from your strict-rubric results. The other numbers are from the version 3 run and should not change, because generation is greedy and unchanged.

```
I used Qwen2.5-3B-Instruct (open-source, instruction-tuned, fits the T4). The prompt sets a persona, rules, fail case and length limit. BLEU/ROUGE check word overlap (0.008/0.11), BERTScore meaning (0.25), a strict 1–5 rubric quality. Strength: respectful replies (tone [X]/5), no refusals. Limitation: generic arguments (relevance [Y]/5).
```

### Version B: long explanation (study notes)

For the generation method I used Qwen2.5-3B-Instruct, an open-source large language model that has been instruction-tuned. This means it was trained further on pairs of instructions and responses, so it follows written instructions reliably. It is a decoder-only model: it writes text one word piece at a time, which is exactly what a generation task needs. An encoder-only model such as BERT cannot generate fluent text. The small models from the course labs were not strong enough; in the lab, the smallest one gave the same answer to every input. I therefore chose a somewhat larger model with three billion parameters and loaded it in 4-bit precision, the same technique as in the course lab, so that it fits on the T4 graphics card together with a larger judge model.

The prompt follows the structure of the starter kit:
- a task;
- a list of rules;
- the hateful comment;
- the label "Counter-narrative:" where the answer should begin.

A system message gives the model the role of an experienced counter-speech operator at an anti-discrimination organisation. The rules ask the model to:
- respond to the specific claim;
- stay calm and respectful;
- use only widely accepted facts, and argue from shared values whenever it is not sure about a fact;
- write two or three sentences;
- output only the reply.

The rules describe what the model should do rather than what it should avoid, because language models tend to do the very thing they are told not to do. I used greedy decoding, which always picks the most likely next word, so the results are reproducible and less likely to contain invented facts.

I evaluated the replies in three ways:
- **Word overlap.** BLEU and ROUGE count the words and word sequences that a reply shares with the human replies. They are standard measures, but strict: a good reply can use completely different words.
- **Meaning.** BERTScore compares the meaning of the words using contextual embeddings, so a reply that says the same thing in other words is still rewarded.
- **A strict rubric scored by a larger judge model.** None of the measures above checks whether a reply is respectful or factual. A larger language model (Qwen2.5-7B-Instruct) therefore scored every reply from one to five on relevance, tone, factuality and clarity, after first writing one sentence about the reply's main weakness.

I also recorded how often the model refused to answer.

**Why the rubric is strict.** In a first run, a one-to-three scale gave every reply full marks, so it could not tell good replies from weaker ones. I therefore made the scale finer, gave each score a precise meaning, told the judge that the top score should be rare, and asked it to criticise before scoring. This is the iterative prompt design taught in the course, applied to the judge.

**Results.**
- Word overlap with the expert replies was very low: BLEU 0.008, ROUGE-1 0.147, ROUGE-2 0.014 and ROUGE-L 0.108.
- BERTScore was 0.254 after rescaling. The replies are clearly closer in meaning to the expert replies than unrelated sentences would be, but they are far from paraphrases of them.
- The model never refused to answer.
- On the strict rubric, the replies scored [fill in: the average for each criterion and overall, and which scores were given most often].

**Strength.** The replies were consistently calm, respectful and fluent, although every input was hateful, and the model never refused. Confirm this with the tone score.

**Limitation.** Many replies were generic. For example, to the claim that most immigrants in the United States are criminals, the model replied that many immigrants contribute positively to society. It did not rebut the claim about crime directly, as the expert did. Confirm this with the relevance score and the judge's critiques.

### Version C: full worked example with real results (strict rubric to be filled in)

**Results (Section 2, zero-shot, 100 test items).** The reference-based numbers are from the version 3 run; the rubric rows come from your version 4 run.

| Metric | Score |
|---|---:|
| BLEU | 0.008 |
| ROUGE-1 | 0.147 |
| ROUGE-2 | 0.014 |
| ROUGE-L | 0.108 |
| BERTScore-F1 (rescaled) | 0.254 |
| Refusal rate | 0.000 |
| Rubric relevance (1–5) | [...] |
| Rubric tone (1–5) | [...] |
| Rubric factuality (1–5) | [...] |
| Rubric clarity (1–5) | [...] |
| Rubric overall (1–5) | [...] |

**Real example (test item 1):**
- *Hate speech:* "Most immigrants in the U.S. are criminals."
- *Human reference:* "Not all immigrants are criminals. And it is important to consider that 13.5% of the US population is composed of migrants that are here legally."
- *Generated (zero-shot):* "Many immigrants contribute positively to society through hard work and innovation. We should focus on their contributions rather than generalizing based on a small minority."
- *Version 3 judge (1–3 scale):* 3 on every criterion.
- *Version 4 judge (1–5 scale):* relevance [...], tone [...], factuality [...], clarity [...]; critique: "[...]".
- *What we would expect from the strict anchors:* relevance about 3, because the reply defends the group in general rather than refuting the crime claim; tone 4–5.

**Interpretation of the reference-based results:**
- **BLEU of 0.008 means almost no shared four-word sequences.** The model words its replies completely differently from the experts. This is expected for open-ended generation, and it is why BLEU alone cannot judge counter-narratives (L7 p.45).
- **ROUGE-1 of 0.147** means about one word in seven is shared. ROUGE-2 (0.014) shows that two-word sequences almost never match.
- **BERTScore of 0.254 (rescaled)** is clearly above 0, so the replies are on the same topic and closer in meaning to the references than unrelated sentences, but they are far from paraphrases of them.
- **A refusal rate of 0%:** safety training did not block this task, even with very offensive inputs.

**How to interpret the strict rubric once you have run it:**
- **Check the score distribution table first.**
  - If scores are spread over 2–5, the judge now separates replies, and the averages are meaningful.
  - If almost everything is still 5, the judge is still lenient. Report that as a limitation of LLM-as-judge evaluation.
- **Expected pattern:** high tone (the replies are polite) and clearly lower relevance (many replies defend the group in general instead of refuting the claim). If your numbers show this, they quantify both the strength and the main weakness of zero-shot prompting (L6 p.53).
- **Read the critiques of the lowest-relevance replies.** They show in plain words *why* a reply was judged weak, and make good examples for your explanation.

**Filled-in short answer:** Version A, with tone and relevance from your rubric table.

---
# Section 3: Improving the Counter-Narrative Generation Method (30 marks)

## 3a: Improve the method (15 marks)

### Choice of improvement: retrieval-augmented few-shot prompting

**Idea:**
- For each test hate speech, **find the 3 most similar hate speeches in the training data**, using sentence embeddings and cosine similarity.
- Put those hate speeches *with their expert-written counter-narratives* into the prompt as examples.
- Then ask for a new counter-narrative.

**The LLM judge is *not* the improvement.** It is part of the evaluation and scores both methods in exactly the same way.

The improvement combines two course techniques:
- **Few-shot prompting** (L6 p.54; L3 p.93): "a few (1–5) examples are added into the prompt; the model can generalise based on the examples".
- **Retrieve-Read RAG** (L7 p.5):
  1. query the knowledge base;
  2. get the top-k results;
  3. prepend them to the question;
  4. generate.
  - Our knowledge base is the training set, as the brief suggests ("training data for … retrieval examples").

**Why this improvement (the motivation):**
1. **Zero-shot weaknesses:** "poorer performance; hard to apply to complex tasks" (L6 p.53). Few-shot examples improve performance and show the expected format (L6 p.54).
2. **Choosing the examples well:** few-shot is "sensitive to the number and choice of examples" (L6 p.54). Retrieval gives *relevant* examples for each input instead of fixed random ones.
   - A hate speech about migrants and benefits gets expert replies to similar claims, with the kinds of arguments the experts used.
3. **Grounding against hallucination:** RAG makes the LLM "less likely to generate incorrect information", and "small models perform similarly to larger LLMs on factual tasks" (L7 p.6). The examples contain real expert arguments, which supports the brief's "fact-bound" requirement.
4. **No training needed:** RAG "uses models off-the-shelf" (L7 p.29), so it fits the inference-only rule.
5. **Supported by guidance:** the brief names semantic retrieval / RAG and few-shot examples, and the tutor listed "adding examples" and "RAG".
   - Classmates reported that RAG is *not required*, and that "a library is fine" if you use it.

**Alternatives considered:**

| Alternative | Why not chosen |
|---|---|
| Changing temperature (allowed by the tutor) | Weakest motivation: higher temperature gives more variety but "more hallucinations" (L7 p.44); it adds no knowledge |
| Chain-of-thought (L6 p.55) | Long outputs and more computation; designed for reasoning problems, not short persuasive replies |
| Fixed random few-shot examples (as in Lab7) | Simpler, but the examples may be about a different target and claim (the choice-sensitivity problem) |
| Web-search RAG (tutor option) | Needs a search API; web text is less reliable than expert-written replies |
| Fine-tuning / soft prompting (L6 p.61) | Not allowed: the assignment is inference only |

**What stays exactly the same (a fair comparison):**
- the model;
- the system prompt;
- the `RULES`;
- greedy decoding;
- `MAX_NEW_TOKENS`;
- the 100 test items;
- the metrics and the judge.

The **only** change is the retrieved examples in the prompt.

### Dense retrieval with Sentence Transformers

**How it works:**
1. **Embed** every training hate speech once (L7 p.14: "calculate the embedding of each document… can be done ahead of time").
   - The embedding model turns each text into a vector of 384 numbers that represents its meaning.
   - L7 p.13 describes a Sentence Transformer that collects the meaning in a special [DOC] token. `all-MiniLM-L6-v2` instead **averages all token vectors** ("mean pooling", L6 p.26). Both give one vector per text.
2. **Embed the query**, i.e. the test hate speech.
3. **Find the most similar training texts by cosine similarity:** q* = argmax_d cos(q, d) (L7 p.14).
   - cos(v, w) = (v · w) / (|v| |w|) (L3 p.31).
   - 1 = same direction (very similar meaning); 0 = unrelated.
   - The standard function `util.semantic_search` computes these similarities and returns the best matches, sorted from best to worst.
   - Checking every training vector is the simple search from L7 p.14. That is fine for about 4,000 training texts; approximate nearest-neighbour search (L7 p.15) is only needed for millions.

**Why dense rather than sparse retrieval:** sparse keyword retrieval (TF-IDF, L4 p.38; L7 p.11) needs the same words to match. Dense retrieval also finds similar meaning with different words (L7 pp.11–12), for example "foreigners" and "immigrants".

**Embedding model `all-MiniLM-L6-v2`:** a small (about 80 MB), fast and widely used general-purpose sentence-embedding model.

**Diversity filter:** in the training data, the same hate speech often appears with several counter-narratives. We skip a retrieved hate speech we have already used, so the 3 examples are about 3 *different* comments. This is a simple version of "filtering for diversity" (L7 p.22, MMR p.23).

In [ ]:
print("Loading embedding model", EMBEDDING_MODEL_NAME, "...")
embedder = SentenceTransformer(EMBEDDING_MODEL_NAME)

train_texts = list(train_df["HATE_SPEECH"])           # the "documents" we search in
train_replies = list(train_df["COUNTER_NARRATIVE"])   # the expert reply for each document

# Embed all training hate speeches once (this is our "knowledge base").
# normalize_embeddings=True gives every vector length 1, which is standard for cosine similarity.
train_embeddings = embedder.encode(train_texts, convert_to_tensor=True,
                                   normalize_embeddings=True, show_progress_bar=True)
embedding_shape = tuple(train_embeddings.shape)
print("Embedding matrix shape (texts, numbers per text):", embedding_shape)

In [ ]:
def retrieve_examples(hate_speech, k=K_EXAMPLES):
    """Return the k most similar TRAINING pairs for a hate speech (dense retrieval, L7 p.14)."""
    query_embedding = embedder.encode(hate_speech, convert_to_tensor=True,
                                      normalize_embeddings=True)

    # semantic_search returns one result list per query; we have one query, so we take [0].
    # We ask for 5 candidates per example we need, because duplicates will be skipped.
    hits = util.semantic_search(query_embedding, train_embeddings, top_k=k * 5)[0]

    examples = []
    used_texts = set()
    for hit in hits:
        position = hit["corpus_id"]             # row number in the training data
        similar_text = train_texts[position]
        if similar_text in used_texts:          # diversity filter: skip a repeated hate speech
            continue
        used_texts.add(similar_text)
        examples.append({
            "hate_speech": similar_text,
            "counter_narrative": train_replies[position],
            "similarity": hit["score"],         # cosine similarity (L3 p.31)
        })
        if len(examples) == k:
            break
    return examples


# Demo: what does retrieval find for the first test item?
demo_item = test_items[0]
demo_examples = retrieve_examples(demo_item["hate_speech"])
print("TEST HATE SPEECH:", demo_item["hate_speech"])
print()
for example in demo_examples:
    print("similarity:", round(example["similarity"], 3))
    print("  similar hate speech:", example["hate_speech"])
    print("  expert reply:       ", example["counter_narrative"])
    print()

### The retrieval-augmented prompt

- It uses the same Task line and the same `RULES` as Section 2. In between there is an "examples" block, introduced as "good counter-narratives written by NGO experts for similar comments" (a context element, L6 p.46).
- We tell the model to use the examples "as a guide for tone, length and kind of arguments, but write a new reply". Otherwise models sometimes just copy an example.
- **The most similar example goes *last*, next to the new comment.** LLMs pay most attention to the beginning and end of the input ("Lost in the Middle", L7 p.21), so the most useful example sits closest to the question.
- The format "Comment: … / Counter-narrative: …" mirrors Lab7's few-shot prompt ("Sentence: … / Category: …") and the SK prompt.

In [ ]:
def build_rag_prompt(hate_speech, target, examples):
    """Build the retrieval-augmented few-shot prompt (Section 3)."""
    target_text = TARGET_DESCRIPTIONS.get(target, "a group of people")

    ordered_examples = list(reversed(examples))     # most similar example last
    examples_text = ""
    for number, example in enumerate(ordered_examples, start=1):
        example_comment = example["hate_speech"]
        example_reply = example["counter_narrative"]
        examples_text += f"Example {number}\n"
        examples_text += f"Comment: {example_comment}\n"
        examples_text += f"Counter-narrative: {example_reply}\n\n"

    prompt = f"""Task: Write a counter-narrative response to the hateful comment below.
The comment targets {target_text}.

Here are examples of good counter-narratives written by NGO experts for similar comments.
Use them as a guide for tone, length and kind of arguments, but write a new reply that fits the new comment.

{examples_text}{RULES}

Comment:
{hate_speech}

Counter-narrative:"""
    return prompt


# Show the full prompt for the first test item
print(build_rag_prompt(demo_item["hate_speech"], demo_item["target"], demo_examples))

In [ ]:
# ---- Generate with the improved method (same loop as Section 2, only the prompt differs) ----
rag_predictions = []
best_similarities = []        # similarity of the best retrieved example (for the analysis)

for item in tqdm(test_items, desc="Retrieval few-shot generation"):
    examples = retrieve_examples(item["hate_speech"])
    prompt = build_rag_prompt(item["hate_speech"], item["target"], examples)
    answer = generate_text(tokenizer, generator, SYSTEM_PROMPT, prompt, MAX_NEW_TOKENS)
    rag_predictions.append(clean_output(answer))
    best_similarities.append(examples[0]["similarity"])

rag_df = make_results_table(test_items, rag_predictions)
rag_df["best_example_similarity"] = best_similarities
rag_df.to_csv("section3_rag_outputs.csv", index=False)
print("Saved section3_rag_outputs.csv")

average_similarity = rag_df["best_example_similarity"].mean()
print(f"Average similarity of the best retrieved example: {average_similarity:.3f}")

## 3b: Evaluate the improved method (10 marks)
*Brief: evaluate the new outputs using the same metrics as Section 2b and display a side-by-side performance comparison table.*

We use **exactly the same functions** as Section 2: `compute_reference_scores`, the refusal column, `judge_all` and `average_rubric_scores`. Lab7 also compared all prompting methods with the same metrics, one after another.

**Extra analyses that make the comparison more convincing:**
- **Difference column (Section 3 − Section 2).** Higher is better for every row **except the refusal rate**, where lower is better.
- **A bar chart** of the reference-based scores.
- **BERTScore per target group:** does retrieval help some groups more than others?
- **Per-example wins:** for how many of the 100 items did BERTScore go up? This shows whether a change in the average comes from many items or just a few.
- **Side-by-side examples:** numbers can hide what changed, so we read some outputs.

In [ ]:
rag_reference_scores, rag_bertscores = compute_reference_scores(rag_predictions, test_items)
rag_refusal_rate = rag_df["refusal"].mean()
rag_judgements, rag_critiques = judge_all(test_items, rag_predictions)

for criterion in RUBRIC:
    rag_df["rubric_" + criterion] = rag_judgements[criterion]
rag_df["rubric_critique"] = rag_critiques
rag_df.to_csv("section3_rag_outputs.csv", index=False)

section3_scores = {}
section3_scores.update(rag_reference_scores)
section3_scores["Refusal rate"] = rag_refusal_rate
section3_scores.update(average_rubric_scores(rag_judgements))

In [ ]:
# ---- Side-by-side comparison table ----
comparison = pd.DataFrame({
    "Section 2: zero-shot": section2_scores,
    "Section 3: retrieval few-shot": section3_scores,
})
comparison["Difference (3 - 2)"] = (comparison["Section 3: retrieval few-shot"]
                                    - comparison["Section 2: zero-shot"])
print("Higher is better for all rows except 'Refusal rate' (lower is better).")
display(comparison.round(3))

# ---- Bar chart of the reference-based scores ----
main_metrics = ["BLEU", "ROUGE-1", "ROUGE-2", "ROUGE-L", "BERTScore-F1"]
method_columns = ["Section 2: zero-shot", "Section 3: retrieval few-shot"]
comparison.loc[main_metrics, method_columns].plot(kind="bar", figsize=(9, 4))
plt.title("Figure 4: Reference-based scores, Section 2 vs Section 3")
plt.ylabel("Score")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# ---- Per-example and per-target analysis (BERTScore F1) ----
per_example = pd.DataFrame({
    "target": rag_df["target"],
    "zero_shot": baseline_bertscores,
    "rag": rag_bertscores,
})
per_example["difference"] = per_example["rag"] - per_example["zero_shot"]

print("Average BERTScore-F1 per target group:")
per_target_scores = per_example.groupby("target")[["zero_shot", "rag", "difference"]].mean()
display(per_target_scores.round(3))

n_improved = (per_example["difference"] > 0).sum()
n_compared = len(per_example)
print(f"Retrieval improved BERTScore for {n_improved} of {n_compared} test items.")

# ---- Strict rubric: score distribution and per-item comparison of relevance ----
print("How often each score was given (Section 3, rows = score 1-5):")
display(score_distribution(rag_judgements))

relevance_difference = rag_judgements["relevance"] - baseline_judgements["relevance"]
n_relevance_up = (relevance_difference > 0).sum()
n_relevance_down = (relevance_difference < 0).sum()
print(f"Strict-rubric relevance went up for {n_relevance_up} items and down for {n_relevance_down} items.")

In [ ]:
# ---- Read a few outputs side by side ----
for i in range(3):
    zero_shot_relevance = baseline_judgements["relevance"][i]
    rag_relevance = rag_judgements["relevance"][i]
    print("=" * 100)
    print("HATE SPEECH:", test_items[i]["hate_speech"])
    print("HUMAN:      ", test_items[i]["references"][0])
    print("ZERO-SHOT:  ", baseline_predictions[i])
    print("   judge:    relevance", zero_shot_relevance, "|", baseline_critiques[i])
    print("RETRIEVAL:  ", rag_predictions[i])
    print("   judge:    relevance", rag_relevance, "|", rag_critiques[i])

### How to interpret the comparison (and what to write if it barely improves)

- **What we expect:**
  - With retrieval, replies usually become **more similar in style and content to the expert replies**, so BERTScore and ROUGE tend to go up.
  - Replies also become more specific: they argue against the claim instead of saying "everyone deserves respect".
- **Caution: style imitation.**
  - The retrieved examples come from the *same dataset* as the references.
  - Part of a higher BLEU/ROUGE may come from copying the dataset's *style*, not from better arguments.
  - Check the rubric (relevance, factuality) and read some outputs before claiming the replies are *better*.
- **If the scores barely change:** classmates reported that RAG "barely moved the dial". That is a valid result. The brief marks "the quality of the improvement, the evaluation, and how clearly you justify and explain your approach", not a guaranteed gain. Possible reasons:
  - examples that were not very similar (check `best_example_similarity`);
  - harsh word-overlap metrics;
  - a small 3B model that does not always make good use of the examples.
- **Limitations to mention:**
  - longer prompts, so slower generation;
  - only 100 test items;
  - a same-family LLM judge (but equal for both methods);
  - BLEU/ROUGE don't measure factuality (L7 p.45);
  - possible copying of statistics from the examples.

## 3c: Explanation (5 marks)

**Question:** Briefly explain your improvement method and whether it improved the counter-narrative generation. Include: what improvement you applied; why you selected it; how the results changed compared with Section 2; one benefit and one limitation of the improved method. (Max **50 words**, in your own words.)

### Version A: short answer (45 words)

Fill in `[A]` and `[B]` (average strict-rubric relevance in Section 2 and Section 3). The other numbers are from the version 3 run.

```
I added retrieval-augmented few-shot prompting: three similar training pairs (average similarity 0.80) guide each reply, because zero-shot replies were generic. BERTScore rose slightly (0.254 to 0.262; 57/100 items); strict-rubric relevance changed from [A] to [B]. Benefit: more specific, expert-like replies. Limitation: replies sometimes copy examples.
```

### Version B: long explanation (study notes)

**What I changed.** I improved the method with retrieval-augmented few-shot prompting, which is a simple form of retrieval-augmented generation. For every test hate speech:
1. A sentence-embedding model turns the comment into a vector.
2. The method finds the three training comments with the highest cosine similarity, skipping a comment that was already chosen.
3. It places those three comments, with their expert-written counter-narratives, in the prompt as examples, with the most similar example last, closest to the new comment.

Everything else stayed exactly the same as in Section 2: the language model, the system message, the rules, greedy decoding, the length limit, the hundred test comments and all evaluation measures. Any difference in the results therefore comes from the examples.

**Why I chose it.** The zero-shot replies were polite but often generic: they defended the group in general instead of answering the specific claim. Examples show the model what a good, specific reply looks like. Few-shot prompting is known to be sensitive to which examples are chosen, and retrieval solves this by picking examples about similar claims. Retrieval also grounds the reply in arguments that experts actually used, which should reduce invented facts, and it needs no training, which the assignment does not allow.

**How the results changed.**
- Retrieval found very close examples: the most similar training comment had an average cosine similarity of 0.80.
- The changes in the reference-based scores were small:
  - BERTScore rose from 0.254 to 0.262, and improved for 57 of the 100 comments.
  - BLEU doubled, from 0.008 to 0.016, but remains tiny.
  - ROUGE-2 and ROUGE-L rose by 0.002 each, while ROUGE-1 fell slightly, from 0.147 to 0.141.
- The improvement is not clearly reliable. If retrieval made no difference at all, an improvement on 57 or more of 100 comments would still happen by chance about one time in ten.
- There were still no refusals.
- On the strict rubric, relevance changed from [fill in] to [fill in]; it went up for [fill in] comments and down for [fill in]. The other criteria changed by [fill in].

**What the replies show.** With retrieval, replies became shorter and more direct. To "Most immigrants in the U.S. are criminals.", the model now answered "There is no evidence that immigrants in the U.S. are more likely to be criminals than native-born citizens." That reply attacks the claim itself, but it is almost a copy of one retrieved expert reply ("There is no evidence that migrants commit more crimes than natives"). This copying of style also explains why BLEU rose while ROUGE-1 fell: shorter replies share more exact word sequences but fewer words in total. Retrieval did not always help: for a comment calling for people of colour to be removed, the retrieval reply ("Addressing systemic issues can lead to better outcomes for everyone, including communities of color.") was vaguer than the zero-shot reply.

**Benefit and limitation.**
- **Benefit:** when a closely matching expert example exists, replies become more specific and closer to how experts answer. Check this against the change in relevance.
- **Limitations:**
  - the gain in the reference-based scores is small and uncertain;
  - the model sometimes copies an example almost word for word, and some replies became too short or vague;
  - the higher overlap scores may partly come from imitating the dataset's style rather than from better arguments.

### Version C: full worked example with real results (strict rubric to be filled in)

**Results (100 test items).** The reference-based numbers are from the version 3 run; the rubric rows come from your version 4 run.

| Metric | Section 2: zero-shot | Section 3: retrieval few-shot | Difference |
|---|---:|---:|---:|
| BLEU | 0.008 | 0.016 | +0.008 |
| ROUGE-1 | 0.147 | 0.141 | −0.007 |
| ROUGE-2 | 0.014 | 0.016 | +0.002 |
| ROUGE-L | 0.108 | 0.110 | +0.002 |
| BERTScore-F1 | 0.254 | 0.262 | +0.008 |
| Refusal rate | 0.000 | 0.000 | 0.000 |
| Rubric relevance (1–5) | [...] | [...] | [...] |
| Rubric tone (1–5) | [...] | [...] | [...] |
| Rubric factuality (1–5) | [...] | [...] | [...] |
| Rubric clarity (1–5) | [...] | [...] | [...] |
| Rubric overall (1–5) | [...] | [...] | [...] |

| Target | BERTScore zero-shot | BERTScore retrieval | Difference |
|---|---:|---:|---:|
| POC | 0.194 | 0.245 | +0.051 |
| MIGRANTS | 0.251 | 0.272 | +0.020 |
| JEWS | 0.238 | 0.248 | +0.009 |
| MUSLIMS | 0.264 | 0.272 | +0.008 |
| DISABLED | 0.211 | 0.213 | +0.001 |
| WOMEN | 0.294 | 0.290 | −0.003 |
| other | 0.216 | 0.208 | −0.008 |
| LGBT+ | 0.240 | 0.226 | −0.015 |

Other results:
- Retrieval improved BERTScore for 57 of 100 items.
- The average similarity of the best retrieved example was 0.802.
- Strict-rubric relevance went up for [...] items and down for [...] items.

**Real examples (test items 1–3):**

| Hate speech | Zero-shot | Retrieval few-shot |
|---|---|---|
| "Most immigrants in the U.S. are criminals." | "Many immigrants contribute positively to society through hard work and innovation. We should focus on their contributions rather than generalizing based on a small minority." | "There is no evidence that immigrants in the U.S. are more likely to be criminals than native-born citizens." |
| "Negroes create social problems so we must get rid of them." | "People of color face systemic challenges, but eradicating entire groups won't solve those issues. Instead, addressing root causes and promoting equality benefits everyone." | "Addressing systemic issues can lead to better outcomes for everyone, including communities of color." |
| "There is an invasion on a level never seen before in history. Millions of people pouring across our borders replacing our race and culture." | "This perspective overlooks the fact that many migrants seek better lives for themselves and their families, just as immigrants have done throughout history. We share a common humanity and deserve to coexist peacefully." | "It's important to remember that migration happens all the time, and it doesn't necessarily mean an invasion. Integration and coexistence can lead to rich cultural exchange." |

The side-by-side cell in version 4 also prints the judge's relevance score and critique for each of these replies. Add them to this table after your run.

**Interpretation of the real results so far:**
- **Retrieval works as intended.** The best example is very close to the test comment (average similarity 0.80; 0.925 for item 1), so the model really gets relevant expert replies.
- **The effect on the reference-based scores is small and uncertain.**
  - BERTScore +0.008 and BLEU +0.008 are tiny absolute changes.
  - 57 of 100 items improving is only slightly better than a coin flip: with no real effect, 57 or more would still happen about one time in ten.
  - This matches classmates' experience that RAG "barely moved the dial".
- **The replies changed in a visible way.**
  - In examples 1 and 3, the retrieval replies answer the claim itself instead of defending the group in general.
  - They are also shorter, which explains why ROUGE-1 fell slightly while BLEU rose.
- **Copying risk.** Reply 1 is almost a paraphrase of the retrieved expert reply. Part of the BLEU gain is style imitation.
- **Not always better.** Reply 2 is vaguer than the zero-shot reply and does not clearly reject the call to "get rid of" people.
- **Per-target differences are noisy,** because each group has only a few of the 100 items.

**How to interpret the strict rubric comparison once you have run it:**
- **If relevance rises clearly** (for example by 0.3 or more, and up for many more items than down), the rubric confirms what the examples suggest: retrieval made replies more specific. This would be stronger evidence than the small BERTScore change.
- **If relevance barely changes,** retrieval changed the style more than the substance. Report this honestly; the brief marks the justification and evaluation, not a guaranteed gain.
- **Watch factuality and clarity.** Copied facts that do not fit the new comment, or over-short replies, can lower them. Check the critiques of the replies whose scores fell.

**Filled-in short answer:** Version A, with the two relevance averages from your comparison table.

---
# Exam revision summary: what this assignment practised

| Concept | Where it is used here | Course source |
|---|---|---|
| Tokens vs. types, tokenization | 1a length statistics | W2 p.8 |
| Preprocessing choices (and when *not* to apply them) | Cleaning cell | L3 p.70 |
| Data leakage, grouped and stratified train/test split | Split by unique hate speech | The brief; standard ML practice |
| Language model = next-token prediction; chain rule | Why decoder-only LLMs generate text | L5 p.12; L6 pp.13–16 |
| Encoder-only vs. decoder-only vs. encoder–decoder | Choice of LLM | L3 p.91; L6 p.19 |
| Instruction tuning, chat templates (hidden tags) | Model choice, `apply_chat_template` | L6 p.43 |
| Quantisation (4-bit / 8-bit) to fit models in GPU memory | `BitsAndBytesConfig` | Lab7 (4-bit), WS6b (8-bit) |
| Pre-train, prompt, predict (no weight updates) | The whole assignment (inference only) | L6 pp.40–41 |
| Prompt elements; AUTOMAT; CO-STAR; positive instructions; fail cases | Prompt design | L6 pp.46–51, 58, 65–69 |
| Output formatting (JSON for downstream processing) | Judge answers | L6 p.58 |
| Zero-shot vs. few-shot vs. chain-of-thought | Section 2 vs. Section 3; alternatives | L6 pp.53–56; L3 p.93; Lab7 |
| Greedy vs. sampling vs. temperature; softmax | Decoding choice | L7 pp.47–50; L6 p.17; WS7b |
| Hallucination (intrinsic / extrinsic) and its causes | "Fact-bound" rules, factuality criterion | L7 pp.43–44 |
| BLEU / ROUGE (overlap, precision / recall), F1 | 2b metrics | Lab7; L4 pp.31–32; L7 p.45 |
| Cosine similarity, embeddings, BERT | BERTScore, retrieval | L3 pp.30–32, 85 |
| Evaluation dimensions, LLM as judge; judge leniency (ceiling effect) and critique-first scoring | Rubric | L7 pp.27, 41; L6 p.55 |
| RAG: Retrieve-Read, dense vs. sparse retrieval, Sentence Transformers | Section 3 | L7 pp.5, 11–14 |
| Reranking ("lost in the middle"), filtering for diversity, MMR | Example order, duplicate filter | L7 pp.21–23 |
| RAG vs. fine-tuning | Why RAG fits "inference only" | L7 p.29 |

---
# Appendix: validation checklist (version 2, still valid for versions 3 and 4)

### A. Code readability and standards

| Check | Result |
|---|---|
| Calculations, function calls or dictionary look-ups inside f-strings | None: every f-string contains only plain variable names (checked automatically; version 1 had 20) |
| Nested f-strings, list comprehensions, generator expressions, lambdas | None (checked automatically) |
| Regular expressions | None (version 1 used one to read the judge's scores; replaced by JSON + `json.loads`) |
| Deep nesting | No block is nested more than 3 levels deep (checked automatically) |
| Standard tools instead of hand-written code | `train_test_split`, `.sample()`, `value_counts(normalize=True)`, `util.semantic_search`, `tqdm`, `json.loads`, `evaluate.load`, Hugging Face `pipeline` |
| Naming and layout | Constants in UPPER_CASE in one settings cell; functions in snake_case with a docstring; imports at the top |
| Reproducibility | One `RANDOM_SEED` for split, sample and libraries; greedy decoding |
| Dry run | Every code cell was executed with small stand-in models and a synthetic dataset to check that the code runs end to end. The real models and the real data could not be run while preparing the notebook. |

### B. Coverage of the brief

| Brief requirement | Where |
|---|---|
| Counter-narrative dataset with hate speech, counter-narrative and target; training data for building and retrieval, test data only for evaluation | Section 1 (dataset choice, split) |
| 1a: target counts, length statistics of both texts | 1a cells |
| 1b: two clear, labelled plots (targets; input vs. response length) | Figures 1–2 (+ Figure 3) |
| 1c, 2c, 3c: explanations of max 50 words | Version A of each answer |
| 2a: open-source instruction LLM, inference only, prompt for non-aggressive, fact-bound replies, test subset | 2a cells |
| 2b: BLEU, ROUGE, BERTScore + rubric-based scores | 2b cells |
| 3a: one improvement (RAG / structured few-shot) | Section 3 (it is both) |
| 3b: the same metrics + a side-by-side comparison table | 3b cells |

### C. Alignment with your other materials

| Source | Guidance | How it is followed |
|---|---|---|
| Tutor (Q&A) | Label counts and **character length** in the analysis | 1a computes words *and* characters |
| Tutor | Open-source LLM; Ollama allowed; small model fine | Open-source Qwen; Hugging Face pipeline (as in Lab7); a small 3B writer that fits the standard T4 GPU |
| Tutor | Specify the output format in the prompt | "2 to 3 sentences… only the counter-narrative" + "Counter-narrative:" |
| Tutor | Reference-based semantic scores + prompt-based rubric scale; human evaluation optional | BERTScore + LLM-judge rubric; no hand scoring |
| Tutor | Improvement options: examples, RAG, web search, temperature; same metrics | Retrieval few-shot (examples + RAG); identical metrics |
| Tutor | Record the refusal rate | `is_refusal`, refusal rate in both tables |
| Peers | Rubric criteria (relevance, counters the claim, tone, clarity), 1–3 scale, LLM judge optional | Relevance (incl. countering the claim), tone, factuality (brief: "fact-bound"), clarity; strict 1–5 scale (own rubric allowed; 1–3 hit the ceiling in the version 3 run) |
| Peers | RAG not required; a library is fine; improvements may be small | Sentence Transformers library; an honest interpretation section |
| Peers | Colab allowed; full runs are slow | Colab T4; a 100-item subset |
| Starter kit | Task / Rules / Comment / "Counter-narrative:" structure; save outputs to CSV; 120 tokens | Same structure, CSV saving, `MAX_NEW_TOKENS = 120`; temperature 0.7 replaced by greedy (explained) |
| Lab7 | 4-bit `BitsAndBytesConfig`, text-generation pipeline, `evaluate` BLEU with lowercasing, few-shot from training rows, comparing methods with the same metrics | All used |
| WS6b / WS7b | Iterative prompting, evaluation on a fixed subset; greedy vs. sampling | Sanity check + fixed subset; greedy decoding explained |
| Lectures | See the "Exam revision summary" above | Every choice has a slide reference |

---
# That's a wrap!

Before submitting your own version:
- run all cells top to bottom with saved outputs;
- replace the `[...]` placeholders;
- write the 50-word answers in your own words.